# Capstone: Predicting Late Payments

## What you'll learn
Nothing new. This pulls the whole stage together into one project. Work each step
yourself, then compare with the worked solution at the end.

## The brief

The credit team wants a ranked list of accounts most likely to pay late next, so they can
call them early. Using `data/customer_accounts.csv`:

1. Load the raw data and hold back 20% as a test set (stratified by `paid_late`).
2. Build a scikit-learn `Pipeline` that fills missing values, encodes categories, and scales numbers.
3. Compare at least two scikit-learn models (for example logistic regression and a random
   forest) with 5-fold cross-validated ROC AUC on the training data.
4. Train one neural network, in **PyTorch or TensorFlow**, on the same training data.
5. Pick the best model and report its test ROC AUC, a confusion matrix, and the recall at
   a threshold that flags about 30% of accounts.
6. List the 10 highest-risk accounts in the test set with their probability.
7. Save the chosen model, and write a short recommendation for the credit team.

## Step 1: load and split

In [ ]:
# Your code

## Step 2: preprocessing pipeline

In [ ]:
# Your code

## Step 3: compare scikit-learn models

In [ ]:
# Your code

## Step 4: a neural network (PyTorch or TensorFlow)

In [ ]:
# Your code

## Step 5: evaluate the best model

In [ ]:
# Your code

## Step 6: the ten highest-risk accounts

In [ ]:
# Your code

## Step 7: save the model

In [ ]:
# Your code

## Your recommendation

_Replace this text: which model would you use, how good is it, which threshold would you
suggest, and what are the main warning signs of a late payer?_

## Worked solution

One complete version. Yours may differ and still be correct.

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, recall_score, ConfusionMatrixDisplay

# 1. Load and split
df = pd.read_csv("data/customer_accounts.csv")
df["region"] = df["region"].str.title()
NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
CATEGORICAL = ["segment", "region"]
X, y = df[NUMERIC + CATEGORICAL], df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Preprocessing
preprocess = ColumnTransformer([
    ("numeric", Pipeline([("fill", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
    ("category", Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                           ("encode", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

# 3. Compare scikit-learn models
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
candidates = {
    "logistic regression": LogisticRegression(),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            random_state=42, n_jobs=-1),
}
scores = {}
for name, model in candidates.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    scores[name] = cross_val_score(pipe, X_train, y_train, cv=folds, scoring="roc_auc").mean()
    print(f"{name:20s} cross-validated ROC AUC {scores[name]:.3f}")

# 4. A PyTorch network on the same preprocessed data
prep = preprocess.fit(X_train)
Xt = torch.tensor(prep.transform(X_train), dtype=torch.float32)
Xv = torch.tensor(prep.transform(X_test), dtype=torch.float32)
yt = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
torch.manual_seed(42)
net = nn.Sequential(nn.Linear(Xt.shape[1], 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt, yt), batch_size=64, shuffle=True)
for _ in range(40):
    net.train()
    for inputs, targets in loader:
        optimizer.zero_grad()
        loss_fn(net(inputs).squeeze(1), targets).backward()
        optimizer.step()
net.eval()
with torch.no_grad():
    nn_probability = torch.sigmoid(net(Xv).squeeze(1)).numpy()
print(f"{'PyTorch network':20s} test ROC AUC {roc_auc_score(y_test, nn_probability):.3f}")

# 5. Evaluate the best scikit-learn model on the test set
best_name = max(scores, key=scores.get)
best = Pipeline([("preprocess", preprocess), ("model", candidates[best_name])]).fit(X_train, y_train)
probability = best.predict_proba(X_test)[:, 1]
threshold = pd.Series(probability).quantile(0.70)          # flags about 30% of accounts
flagged = (probability >= threshold).astype(int)
print(f"\nChosen: {best_name}")
print(f"test ROC AUC {roc_auc_score(y_test, probability):.3f}")
print(f"threshold {threshold:.2f} flags {flagged.mean():.0%} and catches "
      f"{recall_score(y_test, flagged):.0%} of late payers")
ConfusionMatrixDisplay.from_predictions(y_test, flagged, display_labels=["on time", "late"])
plt.title(f"{best_name}, threshold {threshold:.2f}")
plt.show()

# 6. Ten highest-risk accounts
risk = df.loc[X_test.index, ["account_id", "segment", "region", "avg_days_to_pay",
                             "support_tickets", "on_contract"]].assign(probability=probability.round(2))
print(risk.sort_values("probability", ascending=False).head(10).to_string(index=False))

# 7. Save
os.makedirs("models", exist_ok=True)
joblib.dump(best, "models/late_payment_capstone.joblib")
print("\nsaved models/late_payment_capstone.joblib")

## Where to go next

You can now train, evaluate, and compare machine learning models with scikit-learn,
PyTorch, and TensorFlow. To keep growing:

- Try the same workflow on a real (approved) dataset from your own work.
- Explore gradient boosting (`sklearn.ensemble.HistGradientBoostingClassifier`), often the
  strongest model for tabular data.
- Learn how models are monitored after launch: data changes over time, and models need retraining.

That completes the course. Well done.